# Enforcement Paradox POC 1: The Broken Shield (Flux.1-dev on AWS SageMaker)

This notebook runs the end-to-end pipeline on an `ml.g5.xlarge` instance (1 x NVIDIA A10G GPU, 24GB VRAM).
We are cloning **George W Bush** using `black-forest-labs/FLUX.1-dev`, applying **Fawkes-equivalent cloaking**, and proving the bypass.

## 1. Setup Environment
Install all required packages for FLUX.1-dev LoRA training and InsightFace evaluation.

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install diffusers transformers accelerate peft safetensors 
!pip install insightface onnxruntime-gpu huggingface_hub scikit-learn open-clip-torch tqdm

## 2. Hugging Face Login
Required to download the gated `FLUX.1-dev` weights.

In [ ]:
import os
import huggingface_hub

os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"
huggingface_hub.login(token=os.environ["HF_TOKEN"])
print("Logged in to Hugging Face successfully!")

## 3. Clone Repository
Since you are running this on a fresh SageMaker instance, we need the `poc1_shield_bypass` scripts.

In [ ]:
import os
import shutil

# Note: If you already uploaded your project folder to SageMaker, 
# you can skip GitHub cloning and just set your workspace directory here.
        WORKSPACE_DIR = "/home/ec2-user/SageMaker/Encode-London"\n
if not os.path.exists(WORKSPACE_DIR):
    print("You need to upload the project files to this instance, or clone them from GitHub.")
    print("Assuming project files are here for the rest of the script.")
    os.makedirs(WORKSPACE_DIR, exist_ok=True)

%cd {WORKSPACE_DIR}

## 4. Download George W Bush Dataset
Downloading 25 images from the Labeled Faces in the Wild (LFW) dataset.

In [ ]:
!python data/download_faces.py \
  --output data/consenting_subjects/george_w_bush \
  --num_images 25 \
  --person "George W Bush"

## 5. Apply Fawkes-Equivalent Adversarial Cloaking
Running the targeted PGD attack with InsightFace ArcFace.
*(This will be much faster on the GPU than on CPU!)*

In [ ]:
!python poc1_shield_bypass/01_cloak_images.py \
  --input data/consenting_subjects/george_w_bush \
  --output poc1_shield_bypass/cloaked_images/george_w_bush_fawkes \
  --method fawkes --mode mid

## 6. Train LoRA on FLUX.1-dev
Training on the **cloaked** images for 1500 steps.
*(Uses 8-bit Adam to fit inside the 24GB VRAM of the A10G)*

In [ ]:
!pip install bitsandbytes  # Required for 8-bit optimizer

!python poc1_shield_bypass/02_train_lora.py \
  --images poc1_shield_bypass/cloaked_images/george_w_bush_fawkes \
  --output poc1_shield_bypass/loras/george_w_bush_fawkes_1500 \
  --base_model black-forest-labs/FLUX.1-dev \
  --steps 1500 \
  --rank 16 \
  --use_8bit

## 7. Generate Images with Trained LoRA
Generating new photos to see if the cloak protected his identity.

In [ ]:
!python poc1_shield_bypass/03_generate_eval.py \
  --lora poc1_shield_bypass/loras/george_w_bush_fawkes_1500 \
  --output poc1_shield_bypass/results/george_w_bush_fawkes_1500 \
  --num_images 3 \
  --steps 30 \
  --trigger_word "ohwx person"

## 8. Compute Enforcement Paradox Evaluation Metrics
Using ArcFace & CLIP to measure if the identity was successfully cloned despite the cloak.
*(Expectation: ArcFace > 0.45 identity similarity)*

In [ ]:
!python poc1_shield_bypass/04_arcface_similarity.py \
  --generated poc1_shield_bypass/results/george_w_bush_fawkes_1500 \
  --reference data/consenting_subjects/george_w_bush \
  --output poc1_shield_bypass/results/george_w_bush_fawkes_1500/similarity_scores.json \
  --skip_fid